# 02 — Accessibility weighting

Designing and testing the function that turns edge tags (`wheelchair`, `surface`) into a routing weight.

### Download the test area again, with the tag fix in place

Re-downloading Financial District, Toronto with `useful_tags_way`/`useful_tags_node` expanded and a backup Overpass mirror set, so `wheelchair`/`surface` survive into the graph.

In [7]:
import osmnx as ox
import matplotlib

ox.settings.useful_tags_way += ["wheelchair", "surface"]
ox.settings.useful_tags_node += ["kerb", "crossing", "tactile_paving"]
ox.settings.overpass_url = "https://overpass.kumi.systems/api/interpreter"

G = ox.graph_from_place("Financial District, Toronto, Canada", network_type="walk")

attrs = G.edges(data=True)



### Design and test the weighting function

`accessibility_weight` takes an edge's attribute dict and returns a weight: the raw `length` by default, a heavy penalty if `wheelchair: no`, a small discount if `wheelchair: yes/designated`, and a small penalty for rough surfaces. Tested first on toy dictionaries, then on real edges — including one with `wheelchair: designated` to confirm the discount actually applies (18.285 → 16.285).

In [10]:
def accessibility_weight(attrs):
    base = attrs.get("length", 1)
    penalty = 0

    if attrs.get("wheelchair") == "no":
        penalty += 1000
    elif attrs.get("wheelchair") in ("yes", "designated"):
        penalty -= 2

    if attrs.get("surface") in ("gravel", "dirt", "grass", "ground"):
        penalty += 10

    return base + penalty

print(accessibility_weight({"wheelchair": "no"}))
print(accessibility_weight({"surface": "gravel"}))


for i, (u, v, data) in enumerate(G.edges(data=True)):
    print(data.get("length"), "->", accessibility_weight(data))
    if i >= 4:
        break

for u, v, data in G.edges(data=True):
    if data.get("wheelchair") == "designated":
        print(data.get("length"), "->", accessibility_weight(data))
        break
    

1001
11
10.053952309088126 -> 10.053952309088126
7.7769490382060225 -> 7.7769490382060225
6.8168161064781145 -> 6.8168161064781145
1.3928388749238232 -> 1.3928388749238232
25.976030231378747 -> 25.976030231378747
18.285034254006842 -> 16.285034254006842


### Visualize: raw length vs. weighted cost, across every edge

Scatter plot of every edge's raw `length` against its `accessibility_weight`. The dashed line is `y = x` — any point sitting exactly on it got no adjustment at all (the neutral majority). Green points (wheelchair `yes`/`designated`) sit slightly *below* the line — a small, deliberate discount. No red points (`wheelchair: no`) show up in this particular area, which matches what we found earlier: this dataset simply doesn't have any `wheelchair: no` tags here, only `yes`/`designated` and untagged edges.

In [ ]:
import matplotlib.pyplot as plt

lengths, weights, colors = [], [], []
for u, v, data in G.edges(data=True):
    length = data.get("length", 0)
    weight = accessibility_weight(data)
    lengths.append(length)
    weights.append(weight)
    if data.get("wheelchair") in ("yes", "designated"):
        colors.append("#2e7d32")
    elif data.get("wheelchair") == "no":
        colors.append("#c62828")
    else:
        colors.append("#9e9e9e")

plt.figure(figsize=(7, 7))
plt.scatter(lengths, weights, c=colors, alpha=0.6, s=15)
max_val = max(max(lengths), max(weights))
plt.plot([0, max_val], [0, max_val], linestyle="--", color="black", linewidth=1, label="y = x (no change)")
plt.xlabel("Raw length (meters)")
plt.ylabel("Accessibility weight")
plt.title("Raw length vs. accessibility-adjusted weight per edge\n(Financial District, Toronto)")
plt.legend()
plt.tight_layout()
plt.savefig("../visualize/accessibility-weight-scatter.png", dpi=150)
plt.show()